# Задание 03. Поиск похожих молекул: фингерпринты и Tanimoto

Найдём, какое вещество из набора больше всего похоже на ибупрофен по структуре.

## Шаг 1. Набор молекул

Ибупрофен и его «соседи»: аспирин, парацетамол, кетопрофен, напроксен, кофеин.

In [ ]:
from rdkit import Chem

ibu = Chem.MolFromSmiles("CC(C)Cc1ccc(cc1)C(C)C(=O)O")
others = {
    "аспирин": "CC(=O)OC1=CC=CC=C1C(=O)O",
    "парацетамол": "CC(=O)Nc1ccc(O)cc1",
    "кетопрофен": "CC(C(=O)O)c1cccc(c1)C(=O)c1ccccc1",
    "напроксен": "COc1ccc2cc(ccc2c1)C(C)C(=O)O",
    "кофеин": "Cn1c(=O)c2c(ncn2C)n(C)c1=O",
}
mols = {k: Chem.MolFromSmiles(v) for k, v in others.items()}


## Шаг 2. Моргановские фингерпринты

Фингерпринт — «отпечаток» молекулы в виде бит. Считаем для всех.

In [ ]:
from rdkit.Chem import AllChem

def fp(m):
    return AllChem.GetMorganFingerprintAsBitVect(m, radius=2, nBits=1024)

fp_ibu = fp(ibu)
fps = {k: fp(m) for k, m in mols.items()}
print("Фингерпринты посчитаны для", len(fps), "молекул")


## Шаг 3. Таблица сходства (Tanimoto)

Чем ближе к 1, тем похожее.

In [ ]:
from rdkit.DataStructs import TanimotoSimilarity

rows = []
for name, f in fps.items():
    rows.append({"вещество": name, "сходство с ибупрофеном": TanimotoSimilarity(fp_ibu, f)})
df = pd.DataFrame(rows).sort_values("сходство с ибупрофеном", ascending=False)
df


## Шаг 4. Визуализация: ряд молекул

Посмотрите, похожи ли «похожие» структуры визуально.

In [ ]:
from rdkit.Chem import Draw

top = df.head(3)
mols_plot = [ibu] + [mols[n] for n in top["вещество"]]
labels = ["ибупрофен"] + list(top["вещество"])
img = Draw.MolsToGridImage(mols_plot, molsPerRow=2, subImgSize=(250, 200), legends=labels)
img


## Шаг 5. Выводы

Почему кетопрофен и напроксен похожи на ибупрофен? Почему кофеин — нет?

**Выводы:**

- ...